# Proyecto: Clasificación de señales de Kepler con un perceptrón simple

## Equipo Azul
### Daniel Felipe Sua Siempira
### Juan David Moreno D'Aleman

## 1. Problema a abordar

La misión Kepler de la NASA observó el brillo de más de 150.000 estrellas
buscando pequeñas caídas periódicas de luz (tránsitos) causadas por un planeta
que pasa frente a su estrella. Cada señal detectada se registra como un
*Kepler Object of Interest* (KOI). Sin embargo, no toda señal es un planeta:
muchas son **falsos positivos** (binarias eclipsantes, ruido instrumental,
contaminación de estrellas cercanas, etc.).

**Objetivo:** construir un perceptrón simple que, a partir de las características
del tránsito y de la estrella, prediga si una señal corresponde a un
**exoplaneta confirmado** o a un **falso positivo**.

Este problema es relevante porque confirmar planetas requiere análisis y
observaciones costosas; un modelo que priorice las señales más prometedoras
ahorra tiempo de revisión.

## 2. Dataset

- **Fuente:** Kepler Objects of Interest (tabla acumulativa), [Enlace del data set](https://www.kaggle.com/datasets/nasa/kepler-exoplanet-search-results?resource=download).
- **Formato:** CSV tabular, 9.564 filas y 50 columnas.
- **Variable de disposición (`koi_disposition`):** CONFIRMED (2.293),
  FALSE POSITIVE (5.023) y CANDIDATE (2.248).
- Las señales CANDIDATE aún no tienen veredicto, por lo que **no se usan para
  entrenar ni evaluar**. Quedan 7.316 filas etiquetadas
  (31 % confirmados, 69 % falsos positivos).

## 3. Diseño del problema

**Tipo de problema:** clasificación binaria (predicción categórica).

**Variable objetivo (y):**
- `1` = CONFIRMED (exoplaneta confirmado)
- `0` = FALSE POSITIVE

**Características de entrada (X):**

| Grupo    | Variables                                                                |
|----------|--------------------------------------------------------------------------|
| Tránsito | `koi_period`, `koi_duration`, `koi_depth`, `koi_impact`, `koi_model_snr` |
| Planeta  | `koi_prad`, `koi_teq`, `koi_insol`                                       |
| Estrella | `koi_steff`, `koi_slogg`, `koi_srad`, `koi_kepmag`                       |

**Columnas excluidas y por qué:**
- Identificadores y texto (`rowid`, `kepid`, `kepoi_name`, `kepler_name`,
  `koi_tce_delivname`): no describen la física de la señal.
- `koi_pdisposition` y `koi_score`: son salidas de otros clasificadores del
  pipeline de Kepler; usarlas sería **fuga de información** (*data leakage*).
- Columnas `_err1/_err2`: se omiten para mantener el modelo simple
  (`koi_teq_err1/2` además están 100 % vacías).
- `koi_fpflag_*` (banderas de falso positivo): excluirlas
  en el modelo principal y compararlas como experimento adicional, porque
  casi revelan la respuesta.

**Preprocesamiento planeado:**
1. Imputar nulos con la mediana (calculada solo con el conjunto de entrenamiento).
2. Aplicar `log1p` a las variables muy sesgadas (`koi_period`, `koi_depth`,
   `koi_prad`, `koi_insol`).
3. Estandarizar las características con media y desviación del entrenamiento.
4. División entrenamiento/prueba [80 % / 20 %], estratificada por clase.

## 4. Diseño del modelo

Usaremos un **perceptrón simple**: una única neurona (una capa `Dense` de
1 unidad en Keras) que conecta las 12 entradas con la salida.

$$z = \mathbf{w} \cdot \mathbf{x} + b \qquad \hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$$

| Componente           | Elección                                               | Justificación                                                                                                                                           |
|----------------------|--------------------------------------------------------|---------------------------------------------------------------------------------------------------------------------------------------------------------|
| Activación de salida | Sigmoide                                               | Entrega un valor en (0, 1) interpretable como la probabilidad de que la señal sea un exoplaneta confirmado. Es adecuada para clasificación binaria.     |
| Función de pérdida   | Entropía cruzada binaria (BCE)                         | Es la pérdida asociada a la sigmoide en clasificación: penaliza fuertemente las predicciones seguras pero equivocadas y da gradientes bien comportados. |
| Optimizador          | Descenso de gradiente (SGD), tasa de aprendizaje [0.1] | Es el algoritmo de descenso de gradiente visto en el curso.                                                                                             |
| Métricas             | Accuracy, precision, recall, F1 y matriz de confusión  | Las clases están desbalanceadas (31 % / 69 %), por lo que accuracy sola no basta.                                                                       |
| Regla de decisión    | $\hat{y} \ge 0.5 \Rightarrow$ exoplaneta               | Umbral estándar.                                                                                                                                        |

**Limitación asumida:** al ser un modelo lineal, solo puede separar las clases con
un hiperplano. Evaluaremos qué tan bien lo logra y discutiremos qué se ganaría con
redes más profundas (fuera del alcance de este proyecto).

## 5. Qué reutilizamos del Laboratorio 04

| Del laboratorio                                                  | Uso en este proyecto                                 |
|------------------------------------------------------------------|------------------------------------------------------|
| `sigmoid`, `binary_cross_entropy`, `accuracy`, `precision`       | Mismas funciones; se agregan recall y F1.            |
| Modelo Keras (`Dense(1, sigmoid)`, `SGD`, `binary_crossentropy`) | Modelo principal; solo cambia el número de entradas. |
| Clase `Perceptron` (NumPy)                                       | Comparación opcional manual vs. Keras.               |
| Curva de pérdida por época                                       | Evidencia de que el modelo aprende.                  |
| Gráfica de pesos por característica                              | Interpretar qué variables influyen más.              |
| Experimentos de normalización y *learning rate*                  | Justificar el preprocesamiento y la tasa elegida.    |

**Lo nuevo respecto al laboratorio:** división entrenamiento/prueba (en el lab se
evaluó con los mismos datos), preprocesamiento calculado solo con el entrenamiento,
imputación de nulos (en lugar de `dropna`), transformación logarítmica, EDA y
matriz de confusión con recall y F1.